# Phase 1: Behavioral Cloning — SetpointGATv2 Training

## Config

In [ ]:
CONFIG = {
    # ── Paths ──
    "drive_root": "/content/drive/MyDrive",
    "dataset_dir": "dataset/setpoint_mixed_V5_mixed_formations",
    "output_dir": "drone_gnn_results",

    # ── Architecture ──
    "in_channels": 58,
    "hidden_channels": 64,
    "out_channels": 4,
    "edge_dim": 7,
    "heads": 4,
    "num_layers": 3,
    "dropout": 0.05,

    # ── Feature engineering ──
    "raw_frame_dim": 32,
    "yaw_idx_in_frame": 25,
    "cos_sin_indices": [25, 26, 54, 55],
    "yaw_quantile": 0.99,

    # ── Training ──
    "batch_size": 64,
    "epochs": 150,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "grad_clip": 1.0,
    "seed": 42,
}

## 1. Setup & Data Loading

In [ ]:
import sys, os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv

# ── Mount Google Drive ──
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

# ── Add src/ to path so we can import dataloader ──
src_dir = str(Path("../src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from dataloader import load_splits, DatasetNormalizer, normalize_batch

# ── Seeds ──
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ── Paths ──
drive_root = Path(CONFIG["drive_root"])
ds_dir = drive_root / CONFIG["dataset_dir"]
out_dir = drive_root / CONFIG["output_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

# ── Load data ──
train_ds, val_ds, test_ds = load_splits(
    ds_dir.parent / (ds_dir.name + "_train.pt"),
    ds_dir.parent / (ds_dir.name + "_val.pt"),
    ds_dir.parent / (ds_dir.name + "_test.pt"),
)
print(f"Splits: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

# ── Fit normalizer on train & move to device ──
normalizer = DatasetNormalizer.fit(train_ds, CONFIG).to(device)
normalizer.save(out_dir / "normalization_stats.pt")
print(f"y_scale: {normalizer.y_scale}")

# ── DataLoaders ──
BS = CONFIG["batch_size"]
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False)

## 2. Model Architecture

In [ ]:
class SetpointGATv2(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch, edge_dim, heads, num_layers, dropout):
        super().__init__()
        head_dim = hid_ch // heads

        self.conv_first = GATv2Conv(in_ch, head_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.proj_first = nn.Linear(in_ch, hid_ch)
        self.norm_first = nn.LayerNorm(hid_ch)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers - 1):
            self.convs.append(GATv2Conv(hid_ch, head_dim, heads=heads, edge_dim=edge_dim, concat=True))
            self.norms.append(nn.LayerNorm(hid_ch))

        self.head = nn.Linear(hid_ch, out_ch)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        res = self.proj_first(x)
        x = self.norm_first(self.conv_first(x, edge_index, edge_attr) + res)
        x = F.dropout(F.elu(x), p=self.dropout, training=self.training)

        for conv, norm in zip(self.convs, self.norms):
            res = x
            x = norm(conv(x, edge_index, edge_attr) + res)
            x = F.dropout(F.elu(x), p=self.dropout, training=self.training)

        return self.head(x)


model = SetpointGATv2(
    in_ch=CONFIG["in_channels"], hid_ch=CONFIG["hidden_channels"],
    out_ch=CONFIG["out_channels"], edge_dim=CONFIG["edge_dim"],
    heads=CONFIG["heads"], num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Training Stack

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
criterion = nn.MSELoss()

## 4. Training & Validation Loop

In [ ]:
best_val = float("inf")
best_path = out_dir / "best_gatv2.pth"
history = {"train": [], "val": [], "lr": []}

for epoch in range(1, CONFIG["epochs"] + 1):
    # ── Train ──
    model.train()
    t_loss, n = 0.0, 0
    for batch in train_loader:
        batch = normalize_batch(batch.to(device), normalizer)
        loss = criterion(model(batch.x, batch.edge_index, batch.edge_attr), batch.target)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optimizer.step()
        t_loss += loss.item() * batch.num_graphs; n += batch.num_graphs
    t_loss /= n

    # ── Validate ──
    model.eval()
    v_loss, n = 0.0, 0
    with torch.no_grad():
        for batch in val_loader:
            batch = normalize_batch(batch.to(device), normalizer)
            loss = criterion(model(batch.x, batch.edge_index, batch.edge_attr), batch.target)
            v_loss += loss.item() * batch.num_graphs; n += batch.num_graphs
    v_loss /= n

    scheduler.step()
    history["train"].append(t_loss); history["val"].append(v_loss)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    tag = ""
    if v_loss < best_val:
        best_val = v_loss
        torch.save(model.state_dict(), best_path)
        tag = " ★"

    if epoch % 10 == 0 or epoch == 1:
        print(f"E{epoch:3d} | train={t_loss:.6f} val={v_loss:.6f} lr={history['lr'][-1]:.2e}{tag}")

print(f"\nBest val: {best_val:.6f} → {best_path}")
torch.save(history, out_dir / "training_history.pt")

In [ ]:
import matplotlib.pyplot as plt

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(history["train"], label="Train"); a1.plot(history["val"], label="Val")
a1.set(xlabel="Epoch", ylabel="MSE"); a1.legend()
a2.plot(history["lr"]); a2.set(xlabel="Epoch", ylabel="LR")
plt.tight_layout(); fig.savefig(out_dir / "curves.png", dpi=150); plt.show()

## 5. Evaluation & Physical Metrics

In [ ]:
model.load_state_dict(torch.load(best_path, weights_only=True))
model.eval()

preds, trues = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = normalize_batch(batch.to(device), normalizer)
        preds.append(model(batch.x, batch.edge_index, batch.edge_attr))
        trues.append(batch.target)

# Un-scale to physical units
p = torch.cat(preds) * normalizer.y_scale
t = torch.cat(trues) * normalizer.y_scale

mse = ((p - t) ** 2).mean(0)
mae = (p - t).abs().mean(0)

for i, name in enumerate(["ΔX(m)", "ΔY(m)", "ΔZ(m)", "ΔYaw(rad)"]):
    print(f"{name:<12} MSE={mse[i]:.8f}  MAE={mae[i]:.8f}")
print(f"\nTranslation  MSE={mse[:3].mean():.8f}  MAE={mae[:3].mean():.8f}")
print(f"Rotation     MSE={mse[3]:.8f}  MAE={mae[3]:.8f}")

## 6. Detailed Physical Evaluation (MAE + RMSE)

In [ ]:
# ── Load best model ──
model.load_state_dict(torch.load(best_path, weights_only=True, map_location=device))
model.eval()

# ── Collect predictions on test set ──
preds, trues = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = normalize_batch(batch.to(device), normalizer)
        preds.append(model(batch.x, batch.edge_index, batch.edge_attr))
        trues.append(batch.target)

pred_norm = torch.cat(preds)
true_norm = torch.cat(trues)

# ── Inverse normalization: scale back to physical units ──
pred_phys = pred_norm * normalizer.y_scale
true_phys = true_norm * normalizer.y_scale
errors = pred_phys - true_phys

# ── Per-dimension metrics ──
mae_per_dim  = errors.abs().mean(dim=0).cpu()
rmse_per_dim = errors.pow(2).mean(dim=0).sqrt().cpu()

# ── Build results table ──
import pandas as pd

labels = ["dX (m)", "dY (m)", "dZ (m)", "dYaw (rad)"]
df = pd.DataFrame({
    "Dimension": labels,
    "MAE": [f"{v:.6f}" for v in mae_per_dim.tolist()],
    "RMSE": [f"{v:.6f}" for v in rmse_per_dim.tolist()],
    "MAE (cm/mrad)": [
        f"{mae_per_dim[0]*100:.3f} cm",
        f"{mae_per_dim[1]*100:.3f} cm",
        f"{mae_per_dim[2]*100:.3f} cm",
        f"{mae_per_dim[3]*1000:.3f} mrad",
    ],
})

# ── Aggregate translation vs rotation ──
trans_mae  = mae_per_dim[:3].mean().item()
trans_rmse = rmse_per_dim[:3].mean().item()
rot_mae    = mae_per_dim[3].item()
rot_rmse   = rmse_per_dim[3].item()

agg = pd.DataFrame({
    "Group": ["Translation (XYZ)", "Rotation (Yaw)"],
    "MAE": [f"{trans_mae:.6f} m", f"{rot_mae:.6f} rad"],
    "RMSE": [f"{trans_rmse:.6f} m", f"{rot_rmse:.6f} rad"],
    "MAE (human)": [f"{trans_mae*100:.3f} cm", f"{rot_mae*1000:.3f} mrad"],
})

print("Per-Dimension Metrics (Physical Units):")
print(df.to_string(index=False))
print()
print("Aggregate Metrics:")
print(agg.to_string(index=False))
print()
print(f"y_scale used: {normalizer.y_scale.cpu()}")
print(f"Test samples: {pred_norm.shape[0]}")